In [29]:
import pandas as pd

In [30]:
df = pd.read_csv("/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/evaluation_dataset.csv")

In [31]:
df.head()

,id,text
0,/Users/midhunln/Documents/rag20march_with_eval...,Page 1 of 132 \n \n \nMASTER CIRCULAR \n \nSEB...
1,/Users/midhunln/Documents/rag20march_with_eval...,Securities and Exchange Board of India (Issue ...
2,/Users/midhunln/Documents/rag20march_with_eval...,Page 2 of 132 \n \n4. With the issuance of thi...
3,/Users/midhunln/Documents/rag20march_with_eval...,"suffered thereunder, any right, privilege, obl..."
4,/Users/midhunln/Documents/rag20march_with_eval...,regulations and bidding portal. \n7. All liste...


In [32]:
df["id"][0]

'/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master Circular for Issue of Capital and Disclosure Requirements.pdf_0_0'

In [33]:
df.info

<bound method DataFrame.info of                                                      id  \
0     /Users/midhunln/Documents/rag20march_with_eval...   
1     /Users/midhunln/Documents/rag20march_with_eval...   
2     /Users/midhunln/Documents/rag20march_with_eval...   
3     /Users/midhunln/Documents/rag20march_with_eval...   
4     /Users/midhunln/Documents/rag20march_with_eval...   
...                                                 ...   
1103  /Users/midhunln/Documents/rag20march_with_eval...   
1104  /Users/midhunln/Documents/rag20march_with_eval...   
1105  /Users/midhunln/Documents/rag20march_with_eval...   
1106  /Users/midhunln/Documents/rag20march_with_eval...   
1107  /Users/midhunln/Documents/rag20march_with_eval...   

                                                   text  
0     Page 1 of 132 \n \n \nMASTER CIRCULAR \n \nSEB...  
1     Securities and Exchange Board of India (Issue ...  
2     Page 2 of 132 \n \n4. With the issuance of thi...  
3     suffered thereunder, 

In [34]:
df_sampled = df.sample(frac = 0.1)

In [35]:
df_sampled.head()

,id,text
458,/Users/midhunln/Documents/rag20march_with_eval...,paragraph 9 below. \n \n7.2. If a listed entit...
808,/Users/midhunln/Documents/rag20march_with_eval...,(b) From which marginalized /vulnerable groups...
358,/Users/midhunln/Documents/rag20march_with_eval...,6. The notice being sent to the shareholders s...
858,/Users/midhunln/Documents/rag20march_with_eval...,"Act, 1961 (43 of 1961). \nReference: http://eb..."
577,/Users/midhunln/Documents/rag20march_with_eval...,"results for the period from ………… to …………, atta..."


In [36]:
!pip install langchain-ollama


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [37]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

chat = ChatOllama(model = "llama3")

In [38]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

chat = ChatOllama(model="llama3")

def llm_call(text):
    prompt = f"""
You are a query generator. Based on the text below, generate a concise search query
that a user would likely use to find this information. The output must only have the query and nothing else.
It is very important that the output is only the query and nothing else.

Text:
{text}
"""
    response = chat.invoke([HumanMessage(content=prompt)])
    return response.content.strip()


# Apply row-wise correctly
df_sampled["query"] = df_sampled["text"].apply(llm_call)

In [39]:
df_sampled["query"].iloc[2]

'"Proposed RPT disclosure requirements"'

In [40]:
df_sampled.to_csv("eval_with_query.csv", index = False)

In [41]:
df_sampled.head()

,id,text,query
458,/Users/midhunln/Documents/rag20march_with_eval...,paragraph 9 below. \n \n7.2. If a listed entit...,"""lodr regulations non compliance"""
808,/Users/midhunln/Documents/rag20march_with_eval...,(b) From which marginalized /vulnerable groups...,"""marginalized groups procurement percentage"""
358,/Users/midhunln/Documents/rag20march_with_eval...,6. The notice being sent to the shareholders s...,"""Proposed RPT disclosure requirements"""
858,/Users/midhunln/Documents/rag20march_with_eval...,"Act, 1961 (43 of 1961). \nReference: http://eb...","""Income Tax Act 1961 salary definition"""
577,/Users/midhunln/Documents/rag20march_with_eval...,"results for the period from ………… to …………, atta...","""company financial results"""


# Construct the qrels dataset for rankx

In [42]:
qrels = {}

for _, row in df_sampled.iterrows():
    query = row["query"]
    doc_id = row["id"]

    if query not in qrels:
        qrels[query] = {}

    qrels[query][doc_id] = 1  # relevance score

In [43]:
qrels

{'"lodr regulations non compliance"': {'/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master circular for compliance with the provisions of the Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) Regulations, 2015 by listed entities.pdf_52_58': 1},
 '"marginalized groups procurement percentage"': {'/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master circular for compliance with the provisions of the Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) Regulations, 2015 by listed entities.pdf_174_8': 1},
 '"Proposed RPT disclosure requirements"': {'/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master circular for compliance with the provisions of the Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) Regulations, 2015 by listed entities.pdf_18_158': 1},
 '

# Start retriever


In [44]:
!pip install langchain-pinecone


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [45]:
import sys
sys.path.append('/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval')

from dotenv import load_dotenv
load_dotenv("/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/ingestion.env")

from Repositories.pinecone_repository import PineconeRepository

In [46]:
from Repositories.pinecone_repository import PineconeConfig
from src.sentence_transformer_embedding import SentenceTransformerEmbedding
from src.sparse_embedding import SentenceTransformerSparseEmbedding



In [47]:
config = PineconeConfig()

dense_embedding_strategy = SentenceTransformerEmbedding(config)

sparse_embedding_strategy = SentenceTransformerSparseEmbedding(config)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5765.79it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 204/204 [00:00<00:00, 24407.75it/s]
The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` 

In [48]:
import os

In [49]:
pinecone_repository = PineconeRepository(
            api_key=os.getenv("PINECONE_API_KEY"),
            environment=os.getenv("PINECONE_ENVIRONMENT", "us-east-1-aws"),
            dense_embedding_strategy=dense_embedding_strategy,
            sparse_embedding_strategy=sparse_embedding_strategy,
            pinecone_config=config
        )

In [50]:
df_sampled["relevant_docs"] = df_sampled["query"].apply(pinecone_repository.query_vector_store_for_rankx)

In [51]:
df_sampled.head()

,id,text,query,relevant_docs
458,/Users/midhunln/Documents/rag20march_with_eval...,paragraph 9 below. \n \n7.2. If a listed entit...,"""lodr regulations non compliance""",[{'id': '/Users/midhunln/Documents/rag20march_...
808,/Users/midhunln/Documents/rag20march_with_eval...,(b) From which marginalized /vulnerable groups...,"""marginalized groups procurement percentage""",[{'id': '/Users/midhunln/Documents/rag20march_...
358,/Users/midhunln/Documents/rag20march_with_eval...,6. The notice being sent to the shareholders s...,"""Proposed RPT disclosure requirements""",[{'id': '/Users/midhunln/Documents/rag20march_...
858,/Users/midhunln/Documents/rag20march_with_eval...,"Act, 1961 (43 of 1961). \nReference: http://eb...","""Income Tax Act 1961 salary definition""",[{'id': '/Users/midhunln/Documents/rag20march_...
577,/Users/midhunln/Documents/rag20march_with_eval...,"results for the period from ………… to …………, atta...","""company financial results""",[{'id': '/Users/midhunln/Documents/rag20march_...


In [52]:
df_sampled["relevant_docs"].iloc[0]

[{'id': '/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master circular for compliance with the provisions of the Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) Regulations, 2015 by listed entities.pdf_56_72',
  'score': 24.3381386},
 {'id': '/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master circular for compliance with the provisions of the Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) Regulations, 2015 by listed entities.pdf_47_42',
  'score': 23.9980011},
 {'id': '/Users/midhunln/Documents/rag20march_with_eval/Ingestion_plus_Retriever_eval/Documents/Master circular for compliance with the provisions of the Securities and Exchange Board of India (Listing Obligations and Disclosure Requirements) Regulations, 2015 by listed entities.pdf_48_46',
  'score': 23.8759499},
 {'id': '/Users/midhunln/Documents/rag20march_with_e

In [53]:
qrels_dict = {}

for i, row in df_sampled.iterrows():
    qid = str(i)
    true_doc = row["id"]   # single doc id string
    qrels_dict[qid] = {true_doc: 1}

In [54]:
run_dict = {}

for i, row in df_sampled.iterrows():
    qid = str(i)
    
    run_dict[qid] = {
        doc["id"]: float(doc["score"])
        for doc in row["relevant_docs"]
    }

In [56]:
from ranx import Qrels, Run, evaluate

qrels = Qrels(qrels_dict)
run = Run(run_dict)

metrics = [
    "mrr",
    "ndcg@10",
    "recall@5",
    "recall@10",
    "precision@5"
]

results = evaluate(qrels, run, metrics)
print(results)

{'mrr': np.float64(0.09219219219219218), 'ndcg@10': np.float64(0.10942265482505567), 'recall@5': np.float64(0.15315315315315314), 'recall@10': np.float64(0.16216216216216217), 'precision@5': np.float64(0.030630630630630633)}
